# MédiaLens — Entraînement du classifieur d'articles

**Auteur** : Adrien Morel — `adrien.morel@medialens.fr`  
**Date** : Janvier 2024  
**Objectif** : Fine-tuner CamemBERT sur notre corpus presse interne pour classifier automatiquement les articles en 5 catégories.

> ⚠️ **Note pour l'équipe qui reprend ce notebook** (Chloé m'a dit que quelqu'un allait refaire ça en API) :  
> Le modèle final est dans `./medialens_classifier/`. Les vrais poids font ~443 Mo.  
> Je n'ai pas eu le temps de faire des tests unitaires. Les dépendances sont dans `requirements.txt` mais j'ai peut-être oublié des versions. Good luck !  
> — Adrien, 03/11/2024

## 1. Imports et configuration

À noter : j'ai laissé quelques `pip install` en dur dans les cellules parce que c'est plus simple pour reproduire l'environnement depuis zéro. Je sais, c'est pas propre.

In [ ]:
# Installs en dur — à mettre dans requirements.txt proprement
# !pip install transformers==4.36.2
# !pip install torch==2.1.0
# !pip install datasets
# !pip install scikit-learn
# !pip install pandas

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import torch
from transformers import (
    CamembertTokenizer,
    CamembertForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
# Note: j'entraînais sur le GPU de mon Mac M2, le modèle final a été entraîné sur Colab Pro

## 2. Chargement des données

Le corpus complet est dans `data/corpus_presse_2021_2023.csv` (12 847 articles).  
⚠️ **Ce fichier n'est pas dans le repo** — trop lourd (28 Mo). Demandez à Chloé l'accès au Drive partagé.  

Pour tester la pipeline, vous pouvez utiliser `articles_test.csv` (500 articles).

In [ ]:
# Chargement du corpus d'entraînement
# df = pd.read_csv('data/corpus_presse_2021_2023.csv')  # fichier original non fourni

# Pour le workshop, utiliser articles_test.csv
df = pd.read_csv('articles_test/articles_test.csv')
print(f'Dataset chargé : {len(df)} articles')
print(df['categorie'].value_counts())
print()
print('Exemples manquants :', df['texte'].isna().sum())
print('Textes vides :', (df['texte'] == '').sum())
print('Textes N/A :', (df['texte'] == 'N/A').sum())

In [ ]:
# Nettoyage basique des données
# (j'aurais dû faire ça plus proprement mais bon...)

df = df.dropna(subset=['texte'])
df = df[df['texte'].str.len() > 10]  # filtre les textes trop courts
df = df[df['texte'] != 'N/A']

# Concaténation titre + texte pour l'entrée du modèle
# (améliore les performances de ~3 points de F1)
df['input_text'] = df['titre'].fillna('') + ' [SEP] ' + df['texte'].fillna('')
df['input_text'] = df['input_text'].str[:512]  # truncate

print(f'Après nettoyage : {len(df)} articles')
print(df['input_text'].str.len().describe())

## 3. Préparation des labels

Ordre des classes (important pour l'inférence !) :  
`0=culture, 1=economie, 2=faits_divers, 3=politique, 4=sport`  

Cet ordre est fixé alphabétiquement. Ne pas changer sans ré-entraîner le modèle.

In [ ]:
LABEL2ID = {
    'culture': 0,
    'economie': 1,
    'faits_divers': 2,
    'politique': 3,
    'sport': 4
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

df['label'] = df['categorie'].map(LABEL2ID)
print('Distribution des labels:')
print(df['label'].value_counts().sort_index())

# Vérif qu'il n'y a pas de NaN dans les labels
print(f'Labels NaN : {df["label"].isna().sum()}')

## 4. Tokenization

J'utilise le tokenizer de CamemBERT. `max_length=256` au lieu de 512 pour la vitesse — la plupart des articles tiennent en 256 tokens.

In [ ]:
MODEL_NAME = 'camembert-base'
tokenizer = CamembertTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples['input_text'],
        padding='max_length',
        truncation=True,
        max_length=256  # compromis vitesse/performance
    )

# Split train/eval
train_df, eval_df = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df['label']
)
print(f'Train: {len(train_df)} | Eval: {len(eval_df)}')

# Conversion en Dataset HuggingFace
train_dataset = Dataset.from_pandas(train_df[['input_text', 'label']].reset_index(drop=True))
eval_dataset  = Dataset.from_pandas(eval_df[['input_text', 'label']].reset_index(drop=True))

train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset  = eval_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(['input_text'])
eval_dataset  = eval_dataset.remove_columns(['input_text'])
train_dataset = train_dataset.rename_column('label', 'labels')
eval_dataset  = eval_dataset.rename_column('label', 'labels')

print('Tokenization terminée')

## 5. Entraînement

⚠️ **Ne pas ré-exécuter cette cellule** — l'entraînement prend environ 2h sur GPU.  
Le modèle final est déjà dans `./medialens_classifier/`.

In [ ]:
# ⚠️ NE PAS EXÉCUTER — entraînement déjà fait, modèle dans ./medialens_classifier/
# Ce code est conservé pour documentation

if False:  # Mettre True uniquement si vous voulez ré-entraîner

    from sklearn.metrics import f1_score, accuracy_score

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        return {
            'f1_macro': f1_score(labels, predictions, average='macro'),
            'accuracy': accuracy_score(labels, predictions)
        }

    model = CamembertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=5,
        id2label=ID2LABEL,
        label2id=LABEL2ID
    )

    training_args = TrainingArguments(
        output_dir='./medialens_classifier',
        num_train_epochs=4,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        evaluation_strategy='epoch',
        save_strategy='best',
        metric_for_best_model='f1_macro',
        load_best_model_at_end=True,
        fp16=False,
        logging_steps=50,
        report_to='none'  # pas de wandb
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    trainer.save_model('./medialens_classifier')
    tokenizer.save_pretrained('./medialens_classifier')
    print('Entraînement terminé — modèle sauvegardé')

## 6. Chargement du modèle final et évaluation

Chargement depuis `./medialens_classifier/` et évaluation sur `articles_test.csv`.

In [ ]:
# Chargement du modèle depuis le dossier local
# ⚠️ Pour le workshop : vous devrez soit utiliser ce modèle (nécessite les vrais poids .bin)
# soit adapter pour utiliser un modèle depuis HuggingFace Hub

try:
    classifier = pipeline(
        'text-classification',
        model='./medialens_classifier',
        tokenizer='./medialens_classifier',
        device=-1  # CPU — mettre 0 pour GPU
    )
    print('Modèle local chargé avec succès')
except Exception as e:
    print(f'Modèle local non disponible ({e})')
    print('Fallback vers camembert-base depuis HuggingFace Hub...')
    # Alternative pour le workshop :
    # classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')
    classifier = None

In [ ]:
# Évaluation sur articles_test.csv
if classifier is not None:
    test_df = pd.read_csv('articles_test/articles_test.csv')
    test_df = test_df.dropna(subset=['texte'])
    test_df = test_df[test_df['texte'].str.len() > 10]
    test_df = test_df[test_df['texte'] != 'N/A']

    test_df['input'] = test_df['titre'].fillna('') + ' [SEP] ' + test_df['texte'].fillna('')
    test_df['input'] = test_df['input'].str[:512]

    # Inférence par batch pour la vitesse
    BATCH_SIZE = 32
    predictions = []
    for i in range(0, len(test_df), BATCH_SIZE):
        batch = test_df['input'].iloc[i:i+BATCH_SIZE].tolist()
        results = classifier(batch)
        predictions.extend([r['label'] for r in results])

    test_df['prediction'] = predictions

    print('=== Rapport de classification ===')
    print(classification_report(test_df['categorie'], test_df['prediction']))
else:
    print('Classifier non initialisé — voir cellule précédente')

## 7. Test d'inférence simple

Exemple d'utilisation du modèle en inférence directe.

In [ ]:
# Exemples de test rapide
textes_test = [
    "Le Premier ministre a annoncé une réforme du système de retraite lors d'une conférence de presse.",
    "La Bourse de Paris clôture en hausse de 1,2 pourcent, portée par les valeurs bancaires.",
    "L'équipe de France de football s'est qualifiée pour la finale de l'Euro.",
    "Le festival de Cannes a ouvert ses portes avec un film franco-américain en compétition officielle.",
    "Un incendie s'est déclaré dans un entrepôt industriel de la banlieue de Lyon, sans faire de victimes."
]

if classifier is not None:
    for texte in textes_test:
        result = classifier(texte[:512])
        print(f'Texte : {texte[:60]}...')
        print(f'  Prédiction : {result[0]["label"]} (score: {result[0]["score"]:.3f})')
        print()
else:
    print('Classifier non disponible')

## 8. Mesure de latence

Mesure de la latence d'inférence — important pour le SLA de l'API (< 500 ms).

In [ ]:
import time

if classifier is not None:
    texte_bench = "Le ministre de l'Économie a présenté ce matin le projet de loi de finances rectificatif pour l'année en cours, prévoyant une hausse des dépenses publiques dans le domaine de la santé."

    # Warm-up
    _ = classifier(texte_bench)

    # Benchmark
    N_RUNS = 100
    latences = []
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        _ = classifier(texte_bench)
        latences.append((time.perf_counter() - t0) * 1000)

    import statistics
    print(f'Latence sur {N_RUNS} inférences :')
    print(f'  Médiane  : {statistics.median(latences):.1f} ms')
    print(f'  P95      : {sorted(latences)[int(0.95*N_RUNS)]:.1f} ms')
    print(f'  P99      : {sorted(latences)[int(0.99*N_RUNS)]:.1f} ms')
    print(f'  Max      : {max(latences):.1f} ms')
    print()
    print(f'SLA cible : < 500 ms p99')
    print(f'Status    : {"✓ OK" if sorted(latences)[int(0.99*N_RUNS)] < 500 else "✗ HORS SLA"}')
else:
    print('Classifier non disponible')

---

## Notes pour l'équipe qui reprend ce travail

### Ce qui fonctionne bien
- La pipeline titre + texte améliore le F1 d'environ 3 points
- CamemBERT fonctionne mieux que les modèles multilingues pour le français
- La classe `faits_divers` est la plus difficile à classifier (confusions avec politique et économie)

### Ce qui reste à faire
- ❌ Pas de tests unitaires
- ❌ Pas d'API — c'est l'objectif du workshop
- ❌ Pas de monitoring de drift
- ❌ Le `requirements.txt` n'est pas complet (j'ai oublié de fixer `sentencepiece` et `sacremoses`)
- ❌ Le modèle n'a pas été testé sur des articles post-2024

### Contact
Adrien Morel — `adrien.morel@medialens.fr`  
*(Je pars chez Dataiku le 15 novembre — contactez Chloé pour les questions urgentes)*